# Data Profiling: Aroma Jaya Coffee Shop

**Tahap:** Prepare.

**Tujuan:** memeriksa kondisi data mentah secara sistematis untuk menemukan masalah data. Hasil profiling ini menjadi dasar aturan di `data_cleaning.ipynb`.

Pertanyaan yang dijawab

1. Seperti apa bentuk data (baris, kolom, tipe)?
2. Kolom mana yang kosong?
3. Apakah ada baris duplikat?
4. Apakah penulisan teks seragam?
5. Apakah angka (quantity, harga) wajar?
6. Apakah data produk konsisten?
7. Apakah data satu transaksi konsisten?
8. Apakah tanggal dan jam wajar dan lengkap?
9. Jadi, apa saja masalahnya dan berapa banyak? 

In [1]:
import pandas as pd

RAW_PATH = "dataset_pos_aromajaya.csv" 
df = pd.read_csv(RAW_PATH)

## 1. Gambaran umum

Lihat jumlah baris dan kolom, tipe data, contoh isi, dan jumlah nilai unik per kolom. Nilai unik membantu mengenali kolom mana yang berupa kategori (sedikit nilai unik) dan mana yang berupa ID.

In [2]:
print(f"Jumlah baris : {len(df):,}")
print(f"Jumlah kolom : {df.shape[1]}")
print(f"ID transaksi unik: {df['transaction_id'].nunique():,}")
df.head(8)

Jumlah baris : 42,643
Jumlah kolom : 10
ID transaksi unik: 30,000


,transaction_id,transaction_date,transaction_time,store_location,product_id,product_name,product_category,quantity,unit_price,tipe_order
0,TX00001,2026-01-01,07:00:18,Gading Serpong (Residential),106,Roti Bakar Bandung,Snack,1,18000,Dine-in
1,TX00002,2026-01-01,07:01:15,Senopati (Hangout),102,Americano,Coffee,2,25000,Takeaway
2,TX00003,2026-01-01,07:04:37,sudirman (office),101,Es Kopi Susu Gula Aren,Coffee,1,22000,takeaway
3,TX00004,2026-01-01,07:07:02,Gading Serpong (Residential),101,Es Kopi Susu Gula Aren,Coffee,1,22000,Dine-in
4,TX00005,2026-01-01,07:07:38,Gading Serpong (Residential),101,Es Kopi Susu Gula Aren,Coffee,1,22000,Dine-in
5,TX00005,2026-01-01,07:07:38,Gading Serpong (Residential),102,Americano,Coffee,1,25000,Dine-in
6,TX00005,2026-01-01,07:07:38,Gading Serpong (Residential),107,Es Teh Manis,Non-Coffee,4,10000,Dine-in
7,TX00006,2026-01-01,07:18:54,Sudirman (Office),104,Matcha Latte,Non-Coffee,1,28000,Dine-in


In [3]:
ringkas = pd.DataFrame({
    "tipe": df.dtypes.astype(str),
    "nilai_unik": df.nunique(),
    "contoh": df.iloc[0],
})
ringkas

,tipe,nilai_unik,contoh
transaction_id,str,30000,TX00001
transaction_date,str,180,2026-01-01
transaction_time,str,21889,07:00:18
store_location,str,6,Gading Serpong (Residential)
product_id,int64,8,106
product_name,str,8,Roti Bakar Bandung
product_category,str,4,Snack
quantity,int64,7,1
unit_price,int64,7,18000
tipe_order,str,4,Dine-in


## 2. Nilai kosong

Hitung jumlah dan persentase baris kosong di setiap kolom.

In [4]:
kosong = pd.DataFrame({
    "jumlah_kosong": df.isna().sum(),
    "persen": (df.isna().mean() * 100).round(3),
})
kosong[kosong["jumlah_kosong"] > 0]

,jumlah_kosong,persen
store_location,7,0.016
tipe_order,6,0.014


In [5]:
# Lihat baris-baris yang kosong supaya tahu konteksnya
df[df["store_location"].isna() | df["tipe_order"].isna()]

,transaction_id,transaction_date,transaction_time,store_location,product_id,product_name,product_category,quantity,unit_price,tipe_order
950,TX00653,2026-01-04,17:28:02,NaN,101,Es Kopi Susu Gula Aren,Coffee,1,22000,Dine-in
2202,TX01537,2026-01-10,17:48:25,Gading Serpong (Residential),103,Café Latte,Coffee,1,30000,NaN
2450,TX01710,2026-01-11,13:35:25,NaN,101,Es Kopi Susu Gula Aren,Coffee,1,22000,Dine-in
3608,TX02499,2026-01-16,12:54:19,Sudirman (Office),102,Americano,Coffee,1,25000,NaN
5666,TX03940,2026-01-24,20:37:24,NaN,104,Matcha Latte,Non-Coffee,1,28000,Takeaway
13436,TX09412,2026-02-27,08:41:27,Sudirman Office,103,Café Latte,Coffee,1,30000,NaN
14005,TX09823,2026-02-28,20:53:25,NaN,103,Café Latte,Coffee,1,30000,Dine-in
20748,TX14597,2026-03-29,17:43:30,NaN,101,Es Kopi Susu Gula Aren,Coffee,1,22000,Dine-in
21375,TX15026,2026-04-01,11:13:33,NaN,107,Es Teh Manis,Non-Coffee,1,10000,Takeaway
29316,TX20598,2026-05-03,20:56:16,Senopati (Hangout),101,Es Kopi Susu Gula Aren,Coffee,1,22000,NaN


## 3. Duplikat

Duplikat penuh = semua kolom sama persis. Untuk transaksi POS, baris yang sama persis di transaksi, produk, jumlah, dan waktu yang sama hampir pasti tercatat dua kali.

In [6]:
n_dup = df.duplicated().sum()
print(f"Duplikat penuh: {n_dup} baris ({n_dup / len(df):.2%})")

# Tampilkan pasangan duplikat (baris asli dan salinannya)
df[df.duplicated(keep=False)].sort_values(["transaction_id", "product_id"]).head(8)

Duplikat penuh: 35 baris (0.08%)


,transaction_id,transaction_date,transaction_time,store_location,product_id,product_name,product_category,quantity,unit_price,tipe_order
962,TX00661,2026-01-04,17:33:00,sudirman (office),103,Café Latte,Coffee,1,30000,Dine-in
963,TX00661,2026-01-04,17:33:00,sudirman (office),103,Café Latte,Coffee,1,30000,Dine-in
2550,TX01782,2026-01-11,17:41:54,Senopati (Hangout),102,Americano,Coffee,2,25000,Dine-in
2551,TX01782,2026-01-11,17:41:54,Senopati (Hangout),102,Americano,Coffee,2,25000,Dine-in
2658,TX01851,2026-01-11,19:17:32,Sudirman Office,103,Café Latte,Coffee,1,30000,Dine-in
2661,TX01851,2026-01-11,19:17:32,Sudirman Office,103,Café Latte,Coffee,1,30000,Dine-in
3429,TX02377,2026-01-15,13:50:18,Senopati (Hangout),105,Croissant Cokelat,Bakery,1,25000,Takeaway
3432,TX02377,2026-01-15,13:50:18,Senopati (Hangout),105,Croissant Cokelat,Bakery,1,25000,Takeaway


In [7]:
# Apakah duplikat itu produk yang sama tercatat dua kali di transaksi yang sama?
pasangan = df.groupby(["transaction_id", "product_id"]).size()
print("Pasangan transaksi-produk yang muncul lebih dari 1 baris:", int((pasangan > 1).sum()))

Pasangan transaksi-produk yang muncul lebih dari 1 baris: 35


## 4. Kolom teks: apakah penulisannya seragam?

Hitung setiap variasi penulisan

In [8]:
print("store_location:")
print(df["store_location"].value_counts(dropna=False), "\n")
print("tipe_order:")
print(df["tipe_order"].value_counts(dropna=False))

store_location:
store_location
Senopati (Hangout)              13943
Gading Serpong (Residential)     7960
sudirman (office)                7073
Sudirman Office                  6780
Sudirman (Office)                6028
Senopati HANGOUT                  852
NaN                                 7
Name: count, dtype: int64 

tipe_order:
tipe_order
Dine-in     21232
Takeaway    19701
DINE IN       883
takeaway      821
NaN             6
Name: count, dtype: int64


In [19]:
peta_cabang = {"sudirman": "Sudirman (Office)", "senopati": "Senopati (Hangout)", "gading": "Gading Serpong (Residential)"}

def baku_cabang(teks):
    if pd.isna(teks):
        return "(kosong)"
    t = str(teks).lower()
    for kata, nama in peta_cabang.items():
        if kata in t:
            return nama
    return "(tidak dikenali)"

def baku_tipe(teks):
    if pd.isna(teks):
        return "(kosong)"
    t = str(teks).lower().replace(" ", "").replace("-", "")
    return {"dinein": "Dine-in", "takeaway": "Takeaway"}.get(t, "(tidak dikenali)")

varian_cabang = (df.assign(nama_baku=df["store_location"].apply(baku_cabang))
                   .groupby(["nama_baku", "store_location"], dropna=False).size()
                   .rename("baris").reset_index())
varian_tipe = (df.assign(nama_baku=df["tipe_order"].apply(baku_tipe))
                 .groupby(["nama_baku", "tipe_order"], dropna=False).size()
                 .rename("baris").reset_index())
display(varian_cabang)
display(varian_tipe)

,nama_baku,store_location,baris
0,(kosong),NaN,7
1,Gading Serpong (Residential),Gading Serpong (Residential),7960
2,Senopati (Hangout),Senopati (Hangout),13943
3,Senopati (Hangout),Senopati HANGOUT,852
4,Sudirman (Office),Sudirman (Office),6028
5,Sudirman (Office),Sudirman Office,6780
6,Sudirman (Office),sudirman (office),7073


,nama_baku,tipe_order,baris
0,(kosong),NaN,6
1,Dine-in,DINE IN,883
2,Dine-in,Dine-in,21232
3,Takeaway,Takeaway,19701
4,Takeaway,takeaway,821


In [20]:
# Pastikan tidak ada nilai yang tidak dikenali oleh aturan di atas
print("Cabang tidak dikenali:", int((df["store_location"].apply(baku_cabang) == "(tidak dikenali)").sum()))
print("Tipe order tidak dikenali:", int((df["tipe_order"].apply(baku_tipe) == "(tidak dikenali)").sum()))
print("Variasi penulisan cabang:", df["store_location"].nunique(), "-> baku: 3")
print("Variasi penulisan tipe order:", df["tipe_order"].nunique(), "-> baku: 2")

Cabang tidak dikenali: 0
Tipe order tidak dikenali: 0
Variasi penulisan cabang: 6 -> baku: 3
Variasi penulisan tipe order: 4 -> baku: 2


## 5. Kolom angka: apakah wajar?

Statistik dan sebaran `quantity` dan `unit_price`. Nilai di luar pola umum (negatif, nol, atau sangat besar) perlu dicek.

In [11]:
df[["quantity", "unit_price"]].describe()

,quantity,unit_price
count,42643.000000,42643.000000
mean,1.439111,22663.860423
std,2.007424,5617.971223
min,-2.000000,10000.000000
25%,1.000000,22000.000000
50%,1.000000,22000.000000
75%,2.000000,25000.000000
max,99.000000,30000.000000


In [12]:
print("Sebaran quantity (jumlah baris per nilai):")
print(df["quantity"].value_counts().sort_index())

Sebaran quantity (jumlah baris per nilai):
quantity
-2        10
 1     31942
 2      6330
 3      2523
 4      1383
 5       440
 99       15
Name: count, dtype: int64


Sebaran di atas menunjukkan hampir semua baris berisi quantity **1 sampai 5**. Nilai `-2` dan `99` berdiri sendiri, jauh dari pola itu.

In [13]:
tidak_wajar = df[(df["quantity"] <= 0) | (df["quantity"] >= 99)]
print(f"Quantity di luar 1-5: {len(tidak_wajar)} baris")
tidak_wajar.sort_values("quantity").head(12)

Quantity di luar 1-5: 25 baris


,transaction_id,transaction_date,transaction_time,store_location,product_id,product_name,product_category,quantity,unit_price,tipe_order
6799,TX04735,2026-01-30,12:15:38,Senopati (Hangout),102,Americano,Coffee,-2,25000,Dine-in
15237,TX10709,2026-03-06,17:04:05,sudirman (office),107,Es Teh Manis,Non-Coffee,-2,10000,Takeaway
12814,TX08973,2026-02-23,13:35:59,Senopati (Hangout),105,Croissant Cokelat,Bakery,-2,25000,Takeaway
16159,TX11363,2026-03-10,07:51:02,Sudirman Office,101,Es Kopi Susu Gula Aren,Coffee,-2,22000,Takeaway
20144,TX14168,2026-03-27,18:25:19,sudirman (office),105,Croissant Cokelat,Bakery,-2,25000,Takeaway
20453,TX14380,2026-03-28,18:36:20,Gading Serpong (Residential),102,Americano,Coffee,-2,25000,Takeaway
22061,TX15527,2026-04-04,17:50:39,Senopati (Hangout),103,Café Latte,Coffee,-2,30000,Takeaway
38120,TX26784,2026-06-10,12:20:13,Sudirman Office,103,Café Latte,Coffee,-2,30000,Dine-in
39986,TX28114,2026-06-19,07:47:02,Senopati (Hangout),105,Croissant Cokelat,Bakery,-2,25000,Dine-in
30915,TX21750,2026-05-10,21:42:53,Senopati (Hangout),103,Café Latte,Coffee,-2,30000,Takeaway


## 6. Konsistensi produk

Satu `product_id` seharusnya punya tepat satu nama, satu kategori, dan satu harga. Kalau lebih, ada salah input atau perubahan harga yang perlu dipahami dulu.

In [14]:
produk = df.groupby("product_id").agg(
    nama=("product_name", "nunique"),
    kategori=("product_category", "nunique"),
    harga=("unit_price", "nunique"),
    harga_min=("unit_price", "min"),
    harga_maks=("unit_price", "max"),
    baris=("product_id", "size"),
)
produk["nama_produk"] = df.groupby("product_id")["product_name"].first()
produk

,nama,kategori,harga,harga_min,harga_maks,baris,nama_produk
product_id,,,,,,,
101,1,1,1,22000,22000,13467,Es Kopi Susu Gula Aren
102,1,1,1,25000,25000,8387,Americano
103,1,1,1,30000,30000,5512,Café Latte
104,1,1,1,28000,28000,3683,Matcha Latte
105,1,1,1,25000,25000,3256,Croissant Cokelat
106,1,1,1,18000,18000,2799,Roti Bakar Bandung
107,1,1,1,10000,10000,5054,Es Teh Manis
108,1,1,1,20000,20000,485,Kentang Goreng


In [15]:
print("product_id yang punya >1 nama / kategori / harga:",
      int((produk[["nama", "kategori", "harga"]] > 1).any(axis=1).sum()))
print("Nama produk yang punya >1 product_id:",
      int((df.groupby("product_name")["product_id"].nunique() > 1).sum()))

product_id yang punya >1 nama / kategori / harga: 0
Nama produk yang punya >1 product_id: 0


## 7. Konsistensi transaksi

Satu `transaction_id` adalah satu struk. Semua barisnya seharusnya punya tanggal, jam, tipe order, dan **cabang** yang sama. Cek dengan nama yang sudah dibakukan (kalau tidak, variasi penulisan akan terhitung sebagai perbedaan palsu).

In [16]:
cek = df.assign(cabang=df["store_location"].apply(baku_cabang),
                tipe=df["tipe_order"].apply(baku_tipe).replace("(kosong)", pd.NA))   # kosong bukan perbedaan, jadi diabaikan
cek = cek[~cek["cabang"].isin(["(kosong)"])]   # baris tanpa cabang tidak bisa dibandingkan

per_trx = cek.groupby("transaction_id").agg(
    n_tanggal=("transaction_date", "nunique"),
    n_jam=("transaction_time", "nunique"),
    n_tipe=("tipe", "nunique"),
    n_cabang=("cabang", "nunique"),
    n_baris=("product_id", "size"),
)
print("Transaksi dengan >1 tanggal :", int((per_trx["n_tanggal"] > 1).sum()))
print("Transaksi dengan >1 jam     :", int((per_trx["n_jam"] > 1).sum()))
print("Transaksi dengan >1 tipe    :", int((per_trx["n_tipe"] > 1).sum()))
print("Transaksi dengan >1 cabang  :", int((per_trx["n_cabang"] > 1).sum()),
      f"dari {len(per_trx):,} transaksi ({(per_trx['n_cabang'] > 1).mean():.1%})")

Transaksi dengan >1 tanggal : 0
Transaksi dengan >1 jam     : 0
Transaksi dengan >1 tipe    : 0
Transaksi dengan >1 cabang  : 832 dari 29,995 transaksi (2.8%)


In [17]:
# Contoh transaksi yang tercatat di lebih dari satu cabang
multi = per_trx[per_trx["n_cabang"] > 1].index
print("Baris yang terlibat:", int(df["transaction_id"].isin(multi).sum()))
cek[cek["transaction_id"].isin(multi[:3])].sort_values(["transaction_id", "product_id"])[
    ["transaction_id", "transaction_date", "transaction_time", "store_location", "product_name", "quantity"]]

Baris yang terlibat: 2119


,transaction_id,transaction_date,transaction_time,store_location,product_name,quantity
13,TX00008,2026-01-01,07:24:39,sudirman (office),Es Kopi Susu Gula Aren,1
12,TX00008,2026-01-01,07:24:39,Senopati (Hangout),Es Teh Manis,1
16,TX00010,2026-01-01,07:30:43,sudirman (office),Es Kopi Susu Gula Aren,1
17,TX00010,2026-01-01,07:30:43,Senopati HANGOUT,Americano,1
15,TX00010,2026-01-01,07:30:43,Sudirman (Office),Es Teh Manis,1
97,TX00065,2026-01-01,12:12:32,Senopati HANGOUT,Americano,1
96,TX00065,2026-01-01,12:12:32,Sudirman Office,Café Latte,1
95,TX00065,2026-01-01,12:12:32,Sudirman (Office),Matcha Latte,1


In [18]:
# Apakah masalah ini hanya di transaksi multi-baris? (transaksi 1 baris pasti 1 cabang)
print("Transaksi dengan >=2 baris :", int((per_trx["n_baris"] >= 2).sum()))
print("Di antaranya multi-cabang  :", len(multi))

Transaksi dengan >=2 baris : 9021
Di antaranya multi-cabang  : 832


## 8. Tanggal dan jam

Cek apakah format valid, rentang waktunya, dan apakah ada hari atau bulan yang tidak lengkap.

In [21]:
tgl = pd.to_datetime(df["transaction_date"], errors="coerce")
jam = pd.to_datetime(df["transaction_time"], format="%H:%M:%S", errors="coerce")
print("Format tanggal tidak valid:", int(tgl.isna().sum()))
print("Format jam tidak valid    :", int(jam.isna().sum()))
print("Rentang tanggal:", tgl.min().date(), "s.d.", tgl.max().date())
print("Rentang jam    :", df["transaction_time"].min(), "s.d.", df["transaction_time"].max())

Format tanggal tidak valid: 0
Format jam tidak valid    : 0
Rentang tanggal: 2026-01-01 s.d. 2026-06-29
Rentang jam    : 07:00:02 s.d. 21:59:57


In [23]:
# Tanggal yang hilang di dalam rentang
semua = pd.date_range(tgl.min().normalize(), tgl.max().normalize())
ada = set(tgl.dt.normalize())
print("Tanggal hilang di dalam rentang:", [str(d.date()) for d in semua if d not in ada])

# Kelengkapan per bulan: hari yang ada datanya vs jumlah hari dalam bulan
hari = pd.DataFrame({"tgl": tgl.dt.normalize().drop_duplicates()})
hari["bulan"] = hari["tgl"].dt.to_period("M")
per_bulan = hari.groupby("bulan").size().rename("hari_pada_data").to_frame()
per_bulan["hari_dalam_bulan"] = per_bulan.index.days_in_month
per_bulan["kurang_hari"] = per_bulan["hari_dalam_bulan"] - per_bulan["hari_pada_data"]
per_bulan

Tanggal hilang di dalam rentang: []


,hari_pada_data,hari_dalam_bulan,kurang_hari
bulan,,,
2026-01,31,31,0
2026-02,28,28,0
2026-03,31,31,0
2026-04,30,30,0
2026-05,31,31,0
2026-06,29,30,1


In [24]:
# Sebaran transaksi per jam (apakah masuk akal untuk jam buka kedai?)
jam_int = df["transaction_time"].str[:2].astype(int)
df.groupby(jam_int)["transaction_id"].nunique().rename("transaksi").to_frame().T

transaction_time,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21
transaksi,2404,4035,2203,665,659,3087,2151,882,757,1802,2782,2709,2464,2044,1356


## 9. Rangkuman masalah

- Hanya sebagian kecil baris yang **dibuang** (duplikat dan quantity tidak wajar). Sebagian besar masalah berupa **penulisan teks** yang tidak seragam, cukup diseragamkan.
- Data produk (ID, nama, kategori, harga) konsisten, jadi aman dipakai.
- Satu masalah tidak punya jawaban pasti dari data: **transaksi yang tercatat di lebih dari satu cabang**. Keputusannya dicatat di cleaning sebagai keterbatasan.
- Bulan terakhir (Juni) tidak lengkap karena data berakhir sebelum akhir bulan.